# Sentiment Analysis

In this notebook, we test how well four different **machine learning models** can analyze and understand the **sentiment** (positive or negative emotions) expressed in messages on various platforms used by software developers. These platforms include:

- **GitHub** (where developers collaborate on code)
- **Jira** (used for tracking issues and tasks)
- **Mailbox** (for email-based communication)

The models we’re testing are:
- **BERT**
- **XLNet**
- **RoBERTa**
- **ALBERT**

Each model was trained on a mix of data from all three platforms. We then test how well each model performs on new, unseen data from the same platforms.

1. **Overall Accuracy**: How well each model performs across all platforms.
2. **Platform-Specific Accuracy**: How well each model performs on **GitHub**, **Jira**, and **Mailbox** separately.

The results help us understand which models work best across different communication tools and give insights into how sentiment analysis can be applied to real-world developer conversations.

Note: This notebook aims to get the data ready, the `Train.ipynb` notebook trains the models, the `Test.ipynb` tests the models against the datasets.


In [5]:
import os
import sys
sys.path.append(os.path.abspath(".."))
from tabulate import tabulate

from api.filter import *
from api.tokenizer import *
from api.model import *

# Tokenized

This section processes the raw sentiment analysis datasets (`so-dataset.csv`, `gh-dataset.csv`, and `crossplatform_sf_dataset.csv`) by applying a custom text transformation function. The goal is to standardize and clean the text data before training. You can change the transform_text function to specific needs.

There are aditional functions provided in [filter.py](../api/filter.py) and [tokenizer.py](../api/tokenizer.py) that can be used for specific use cases. 

In [ ]:
current_directory = os.getcwd()
root = os.path.abspath(os.path.join(current_directory, "..", ".."))

# Define input dataset paths
input_paths = [
    f"{root}/so-dataset.csv",
    f"{root}/gh-dataset.csv",
    f"{root}/crossplatform_sf_dataset.csv"
]

# Define the text transformation function (ensure transform_text is correctly implemented)
def transform_text(row):
    # Modify this function according to your needs
    # Example: return the original text and a dummy replacement count
    return row["Text"], 1

# Loop through each dataset and process it
for input_path in input_paths:
    # Generate output file name
    output_filename = os.path.splitext(os.path.basename(input_path))[0] + "_tokenized.csv"
    output_path = os.path.join(os.path.dirname(input_path), output_filename)

    # Load dataset
    df = pd.read_csv(input_path)
    print(f"Processing dataset: {input_path}")
    print(df.head())  # Print first few rows for verification

    # Apply text transformation
    df[["Text", "replaced_token"]] = df.apply(transform_text, axis=1, result_type="expand")

    # Calculate total replacements from `replaced_token` column
    total_replacements = df["replaced_token"].sum()

    # Save processed dataset
    df.to_csv(output_path, header=True, index=False)

    print(f"Tokenized dataset saved to: {output_path}\n")


Now that the data is cleaned and tokenized lets look at the datasets we have. 

# Dataset Overview: 

`so-dataset.csv` : Contains Stack Overflow comment data.


 `gh-dataset.csv` : Contains GitHub Stack overflow comment data. 




`crossplatform_sf_dataset.csv`

This dataset is designed for **Software Development Sentiment Classification**, containing user comments or discussions from different platforms with sentiment labels.

## **Column Descriptions**
- **`Text`**: The user comment or discussion content.  
- **`Polarity`**: Sentiment label indicating the emotional tendency of the text:  
  - `2`: Negative sentiment  
  - `0`: Neutral sentiment  
  - `1`: Positive sentiment  
- **`Platform`**: The source platform of the data, indicating where the comment or discussion originated:  
  - `0`: **GitHub** (Discussions related to open-source projects, Issues, Pull Requests)  
  - `1`: **Jira** (Bug reports, task comments in software development management tools)  
  - `2`: **Mailbox** (Developer communication through emails)  

## **Dataset Distribution**
The dataset consists of data from **GitHub, Jira, and Mailbox**, with different sentiment (`Polarity`) distributions across platforms. It can be used to train and evaluate sentiment classification models to analyze developer emotions on different platforms.  


In [ ]:

# Load dataset
input_path = f"{root}/crossplatform_sf_dataset.csv"
df = pd.read_csv(input_path)

# Compute dataset statistics
total_samples = len(df)
polarity_counts = df["Polarity"].value_counts().sort_index()
platform_counts = df["Platform"].value_counts().sort_index()

# Compute Polarity distribution within each Platform
platform_polarity_counts = df.groupby(["Platform", "Polarity"]).size().unstack().fillna(0)

# Print results with formatting
print("=" * 50)
print(f"📊 Dataset Information: cf-dataset.csv")
print("=" * 50)
print(f"Total Samples: {total_samples}\n")

# Polarity distribution
print("📌 Polarity Distribution:")
print(tabulate(polarity_counts.reset_index(), headers=["Polarity", "Count"], tablefmt="pretty"))
print("\n")

# Platform distribution
print("📌 Platform Distribution:")
print(tabulate(platform_counts.reset_index(), headers=["Platform", "Count"], tablefmt="pretty"))
print("\n")

# Platform-wise Polarity distribution
print("📌 Platform-wise Polarity Distribution:")
print(tabulate(platform_polarity_counts, headers="keys", tablefmt="pretty"))
print("=" * 50)


Now that we have an understanding of the three datasets we can move over to the [Train.ipynb](./Train.ipynb) Notebook to start training the models based on the datasets we just prepared and reviewed.

After the models are trained they can be tested with the [Test.ipynb](./Test.ipynb) Notebook. 